# Step 1: Import required libraries

In [ ]:
# Import required libraries
from datasets import load_dataset
from tqdm.auto import tqdm
import time
from run import load_benchmark_data
from huggingface_hub import login
import os
import shutil

# Login with your HuggingFace token
login(token="HF_Token")

print("✓ HuggingFace login successful")

/Users/harrylyu/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/harrylyu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /Users/harrylyu/.cache/huggingface/token
Login successful
✓ HuggingFace login successful


### Run this is have corrupted caches (MMLU-Pro fix)

In [2]:
# Only run this cell if you have corrupted caches

print("🧹 Clearing potentially corrupted dataset caches...")

# Clear MMLU-Pro cache (known corruption issue)
mmlu_cache = os.path.expanduser("~/.cache/huggingface/datasets/TIGER-Lab___mmlu-pro")
if os.path.exists(mmlu_cache):
    shutil.rmtree(mmlu_cache)
    print(f"  ✓ Deleted: {mmlu_cache}")
else:
    print(f"  ℹ️  No cache found for MMLU-Pro")

# Optional: Clear all dataset caches (nuclear option)
# Uncomment if you want to start completely fresh
# cache_dir = os.path.expanduser("~/.cache/huggingface/datasets")
# if os.path.exists(cache_dir):
#     shutil.rmtree(cache_dir)
#     print(f"  ✓ Deleted entire cache: {cache_dir}")

print("\n✓ Cache cleanup complete")

🧹 Clearing potentially corrupted dataset caches...
  ℹ️  No cache found for MMLU-Pro

✓ Cache cleanup complete


### Install missing dependencies for benchmarks

In [3]:
# Install missing dependencies
import subprocess
import sys

print("📦 Installing missing dependencies...")

dependencies = [
    'langdetect',  # Required for ifeval
]

for dep in dependencies:
    try:
        __import__(dep)
        print(f"  ✓ {dep} already installed")
    except ImportError:
        print(f"  ⬇️  Installing {dep}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])
        print(f"  ✓ {dep} installed successfully")

print("\n✓ All dependencies installed")

📦 Installing missing dependencies...
  ✓ langdetect already installed

✓ All dependencies installed


# Step 2: Pre-load benchmarks

In [2]:
# Define benchmarks to test
benchmarks = ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']

print(f"📦 Pre-loading and verifying {len(benchmarks)} datasets...")
print("=" * 60)

# Track results
results = {
    'success': [],
    'failed': [],
    'warnings': []
}

for i, bench in enumerate(benchmarks, 1):
    print(f"\n[{i}/{len(benchmarks)}] Loading {bench.upper()}...")
    print("-" * 60)
    
    try:
        # Load the dataset
        bench_data = load_benchmark_data(bench)
        
        # Get dataset size
        data_size = len(bench_data)
        
        # Verify it's not empty
        if data_size == 0:
            print(f"  ⚠️  WARNING: {bench} loaded but contains 0 items!")
            results['warnings'].append(bench)
        else:
            print(f"  ✓ Successfully loaded {bench}")
            print(f"  ✓ Dataset size: {data_size:,} items")
            
            # Show a sample (first item)
            print(f"  ✓ Sample item keys: {list(bench_data[0].keys())[:5]}...")
            
            results['success'].append(bench)
        
    except Exception as e:
        print(f"  ❌ FAILED to load {bench}")
        print(f"  ❌ Error: {str(e)[:200]}")
        results['failed'].append((bench, str(e)))

print("\n" + "=" * 60)
print("📊 DATASET VERIFICATION SUMMARY")
print("=" * 60)

print(f"\n✅ Successfully loaded ({len(results['success'])}/{len(benchmarks)}):")
for bench in results['success']:
    print(f"  • {bench}")

if results['warnings']:
    print(f"\n⚠️  Warnings ({len(results['warnings'])}):")
    for bench in results['warnings']:
        print(f"  • {bench} - Dataset is empty!")

if results['failed']:
    print(f"\n❌ Failed to load ({len(results['failed'])}):")
    for bench, error in results['failed']:
        print(f"  • {bench}")
        print(f"    Error: {error[:100]}...")
else:
    print("\n🎉 All datasets loaded successfully!")

print("\n" + "=" * 60)

# Final decision
if len(results['failed']) == 0:
    print("\n✅ READY TO PROCEED WITH BATCH SUBMISSION")
    print("   All datasets are downloaded and verified.")
    print("   You can now run the batch submission cell.")
else:
    print("\n⚠️  FIX ERRORS BEFORE PROCEEDING")
    print("   Some datasets failed to load.")
    print("   Please fix the errors above before submitting batches.")

📦 Pre-loading and verifying 6 datasets...

[1/6] Loading MATH...
------------------------------------------------------------
  ✓ Successfully loaded math
  ✓ Dataset size: 1,324 items
  ✓ Sample item keys: ['problem', 'level', 'solution', 'type']...

[2/6] Loading MUSR...
------------------------------------------------------------
  ✓ Successfully loaded musr
  ✓ Dataset size: 756 items
  ✓ Sample item keys: ['narrative', 'question', 'choices', 'answer_index', 'answer_choice']...

[3/6] Loading GPQA...
------------------------------------------------------------
  ✓ Successfully loaded gpqa
  ✓ Dataset size: 448 items
  ✓ Sample item keys: ['Pre-Revision Question', 'Pre-Revision Correct Answer', 'Pre-Revision Incorrect Answer 1', 'Pre-Revision Incorrect Answer 2', 'Pre-Revision Incorrect Answer 3']...

[4/6] Loading MMLU-PRO...
------------------------------------------------------------
  ✓ Successfully loaded mmlu-pro
  ✓ Dataset size: 12,032 items
  ✓ Sample item keys: ['question_

### Check if all benchmark works

In [3]:
# Test that all benchmark-specific imports work
print("🧪 Testing benchmark-specific utilities...")
print("=" * 60)

test_results = {}

# Test ifeval
try:
    from ifeval.utils import process_results
    print("  ✓ ifeval.utils imports successfully")
    test_results['ifeval'] = True
except Exception as e:
    print(f"  ❌ ifeval.utils import failed: {e}")
    test_results['ifeval'] = False

# Test math
try:
    from mathbench.utils import doc_to_text
    print("  ✓ mathbench.utils imports successfully")
    test_results['math'] = True
except Exception as e:
    print(f"  ❌ mathbench.utils import failed: {e}")
    test_results['math'] = False

# Test musr
try:
    from musr.utils import doc_to_text
    print("  ✓ musr.utils imports successfully")
    test_results['musr'] = True
except Exception as e:
    print(f"  ❌ musr.utils import failed: {e}")
    test_results['musr'] = False

# Test gpqa
try:
    from gpqa.utils import process_docs, doc_to_text
    print("  ✓ gpqa.utils imports successfully")
    test_results['gpqa'] = True
except Exception as e:
    print(f"  ❌ gpqa.utils import failed: {e}")
    test_results['gpqa'] = False

# Test bbh
try:
    from bbh.utils import doc_to_text
    print("  ✓ bbh.utils imports successfully")
    test_results['bbh'] = True
except Exception as e:
    print(f"  ❌ bbh.utils import failed: {e}")
    test_results['bbh'] = False

# Test mmlu-pro
try:
    from mmlupro.utils import doc_to_text, doc_to_choice
    print("  ✓ mmlupro.utils imports successfully")
    test_results['mmlu-pro'] = True
except Exception as e:
    print(f"  ❌ mmlupro.utils import failed: {e}")
    test_results['mmlu-pro'] = False

print("\n" + "=" * 60)

# Summary
failed_tests = [k for k, v in test_results.items() if not v]
if failed_tests:
    print(f"\n⚠️  {len(failed_tests)} benchmark utilities failed to import:")
    for test in failed_tests:
        print(f"  • {test}")
else:
    print("\n✅ All benchmark utilities import successfully!")

🧪 Testing benchmark-specific utilities...
  ✓ ifeval.utils imports successfully
  ✓ mathbench.utils imports successfully
  ✓ musr.utils imports successfully
  ✓ gpqa.utils imports successfully
  ✓ bbh.utils imports successfully
  ✓ mmlupro.utils imports successfully


✅ All benchmark utilities import successfully!


### Final confirmation before running

In [4]:
# Final confirmation before proceeding
print("✅ PRE-FLIGHT CHECKLIST")
print("=" * 60)

checklist = {
    "Datasets downloaded": len(results['failed']) == 0,
    "Dependencies installed": all(test_results.values()),
    "API keys configured": True,  # Assuming they're in config.py
}

all_clear = all(checklist.values())

for item, status in checklist.items():
    status_icon = "✅" if status else "❌"
    print(f"{status_icon} {item}")

print("=" * 60)

if all_clear:
    print("\n🚀 ALL SYSTEMS GO!")
    print("   You are ready to run batch submissions.")
    print("   Proceed to the batch submission cell.")
else:
    print("\n⚠️  NOT READY YET")
    print("   Please resolve the issues above before proceeding.")

print("\n" + "=" * 60)

✅ PRE-FLIGHT CHECKLIST
✅ Datasets downloaded
✅ Dependencies installed
✅ API keys configured

🚀 ALL SYSTEMS GO!
   You are ready to run batch submissions.
   Proceed to the batch submission cell.



# Step 3: Clean up NA files or results rejected due to safety issues. (Only run this after you finish Step 5 and identify missing results.)

### Check for NA files (Gemini)

In [4]:
import json
from pathlib import Path

model_name = "gemini-2.5-pro".replace("/", ".").replace("_", ".")
deleted = 0

for bench in ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']:
    bench_name = bench.replace("/", ".").replace("_", ".")
    results_dir = Path(f"results/{bench_name}/{model_name}")
    
    if results_dir.exists():
        for json_file in results_dir.glob("*.json"):
            with open(json_file, 'r') as f:
                data = json.load(f)
            if data.get('model_response') == "NA":
                json_file.unlink()
                deleted += 1

print(f"Deleted {deleted} NA files. Restart kernel and re-run.")

Deleted 0 NA files. Restart kernel and re-run.


### Check for NA files (Claude batch submission)

In [ ]:
# Load batch state
with open('batch_state.json', 'r') as f:
    batch_state = json.load(f)

# Models to clear
models_to_clear = [
    'claude-sonnet-4-20250514',
    'claude-opus-4-1-20250805',
    'claude-3-7-sonnet-20250219'
]

# Benchmarks
benchmarks = ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']

cleared_count = 0

for model in models_to_clear:
    print(f"\n🧹 Clearing state for {model}:")
    
    for bench in benchmarks:
        # Create state key (same logic as in run.py)
        state_key = f"{model}_{bench}".replace("/", "_").replace(".", "_")
        
        if state_key in batch_state.get('anthropic', {}):
            print(f"  ✓ Cleared: {bench}")
            del batch_state['anthropic'][state_key]
            cleared_count += 1
        else:
            print(f"  ℹ️  Not found: {bench}")

# Save updated batch state
with open('batch_state.json', 'w') as f:
    json.dump(batch_state, f, indent=2)

print(f"\n{'='*60}")
print(f"✅ Cleared {cleared_count} batch states")
print(f"{'='*60}")
print("\n📋 Next steps:")
print("1. Re-run batch submission")
print("2. New batches will be created for these models")
print("3. Wait 1-24 hours for completion")
print("4. Re-run again to retrieve results")

### Check for blocked responses

In [5]:
import json
from pathlib import Path

model = 'gemini-2.5-pro'
model_name = model.replace("/", ".").replace("_", ".")
benchmarks = ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']

print(f"🧹 Cleaning blocked responses for {model}")
print("=" * 60)

total_deleted = 0

for bench in benchmarks:
    bench_name = bench.replace("/", ".").replace("_", ".")
    results_dir = Path(f"results/{bench_name}/{model_name}")
    
    if not results_dir.exists():
        print(f"\n{bench}: No results directory")
        continue
    
    deleted_count = 0
    
    for json_file in results_dir.glob("*.json"):
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            # Delete if blocked by safety filter
            if data.get('model_response') == "[BLOCKED_BY_SAFETY_FILTER]":
                print(f"  Deleting: {json_file.name}")
                json_file.unlink()
                deleted_count += 1
                total_deleted += 1
                
        except Exception as e:
            print(f"  ⚠️  Error processing {json_file.name}: {e}")
    
    if deleted_count > 0:
        print(f"\n{bench.upper()}: Deleted {deleted_count} blocked files")
    else:
        print(f"\n{bench.upper()}: No blocked files found")

print("\n" + "=" * 60)
print(f"✅ Total deleted: {total_deleted} blocked responses")
print("=" * 60)

if total_deleted > 0:
    print("\n📋 Next steps:")
    print("1. Re-run your processing cell (Cell 9 or equivalent)")
    print("2. Gemini will retry these items")
    print("3. Expect similar block rates (~20%)")
    print("\n💡 Recommendation: Switch to gemini-2.5-flash instead!")
else:
    print("\nNo blocked files to delete.")

🧹 Cleaning blocked responses for gemini-2.5-pro
  Deleting: math_gemini-2.5-pro_1229_1.json
  Deleting: math_gemini-2.5-pro_1203_0.json
  Deleting: math_gemini-2.5-pro_1270_1.json
  Deleting: math_gemini-2.5-pro_722_1.json
  Deleting: math_gemini-2.5-pro_641_1.json
  Deleting: math_gemini-2.5-pro_1283_1.json
  Deleting: math_gemini-2.5-pro_775_0.json
  Deleting: math_gemini-2.5-pro_775_1.json
  Deleting: math_gemini-2.5-pro_1283_0.json
  Deleting: math_gemini-2.5-pro_641_0.json
  Deleting: math_gemini-2.5-pro_722_0.json
  Deleting: math_gemini-2.5-pro_1203_1.json
  Deleting: math_gemini-2.5-pro_1229_0.json
  Deleting: math_gemini-2.5-pro_628_1.json
  Deleting: math_gemini-2.5-pro_481_1.json
  Deleting: math_gemini-2.5-pro_815_1.json
  Deleting: math_gemini-2.5-pro_454_0.json
  Deleting: math_gemini-2.5-pro_895_0.json
  Deleting: math_gemini-2.5-pro_307_1.json
  Deleting: math_gemini-2.5-pro_682_1.json
  Deleting: math_gemini-2.5-pro_548_1.json
  Deleting: math_gemini-2.5-pro_1323_0.jso

# Step 4: Run the evaluation

In [ ]:
# ONLY RUN THIS CELL AFTER ALL CHECKS PASS ABOVE

from run import run
import time

# Define your models and benchmarks
models = [
    'gemini-2.5-pro'
    # 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'meta-llama/Llama-3.1-8B-Instruct','meta-llama/Llama-3.1-70B-Instruct',
    # 'gpt-5-nano-2025-08-07', 'gpt-5-2025-08-07-thinking', 'gpt-4o-mini-2024-07-18', 'gpt-3.5-turbo-1106',
    # 'gpt-4-0613', 'gpt-4o-2024-11-20', 'o4-mini-2025-04-16', 'o3-2025-04-16', 
    # 'gpt-5-mini-2025-08-07', 'gpt-5-2025-08-07', 'claude-opus-4-1-20250805', 'gemini-2.5-flash'
    # 'mistral-medium-2505', 'Qwen/Qwen3-14B', 'gemini-2.5-flash-lite', 'Llama-3.1-405B-Instruct'
    # 'claude-3-5-haiku-20241022', 'claude-3-haiku-20240307', 'meta-llama/llama-4-scout', 'meta-llama/llama-4-maverick',
    # 'Qwen/Qwen2-7B-Instruct', 'ibm-granite/granite-3.3-8b-base', 'Qwen/Qwen3-32B', 'deepseek-ai/DeepSeek-R1', 
    # 'deepseek-ai/DeepSeek-V3', 'meta-llama/Llama-2-70b-chat-hf', 'meta-llama/Llama-2-7b-chat-hf',  'gemma-3-1b-it',
    # 'ibm-granite/granite-3.3-2b-base', 'claude-3-7-sonnet-20250219', 'claude-sonnet-4-20250514', 'Qwen/Qwen2-72B-Instruct'
    # 'Qwen/Qwen3-235B-A22B-Instruct-2507', 'Qwen/QwQ-32B'
]

benchmarks = ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']

# Create list of all model-benchmark combinations
runs = []
for m in models:
    for b in benchmarks:
        runs.append([m, b])

print(f"🚀 STARTING BATCH SUBMISSION")
print(f"   Total combinations: {len(runs)}")
print("=" * 60)

# Sequential submission with delays
for i, r in enumerate(runs, 1):
    print(f"\n[{i}/{len(runs)}] Processing {r[0]} on {r[1]}...")
    print("-" * 60)
    
    try:
        run(r[1], r[0], reps=2, n=100, use_batch_api=True)
        time.sleep(5)  # Avoid rate limits
        print(f"✓ Submission successful")
        
    except Exception as e:
        print(f"❌ Failed: {e}")
        import traceback
        traceback.print_exc()
        
        # Ask whether to continue
        user_input = input("\n⚠️  Continue with remaining submissions? (y/n): ")
        if user_input.lower() != 'y':
            print("Stopping batch submissions.")
            break
        continue

print("\n" + "=" * 60)
print("✅ ALL SUBMISSIONS COMPLETE!")
print("   Batches are now processing on provider servers.")
print("   Run this cell again later to check status and retrieve results.")
print("=" * 60)

# Step 5: Check the results

In [ ]:
# ============================================================
# VERIFY RESULTS WERE SAVED
# ============================================================

import os
from pathlib import Path
import json

models = [
    'gemini-2.5-pro'
    # 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'meta-llama/Llama-3.1-8B-Instruct','meta-llama/Llama-3.1-70B-Instruct',
    # 'gpt-5-nano-2025-08-07', 'gpt-5-2025-08-07-thinking', 'gpt-4o-mini-2024-07-18', 'gpt-3.5-turbo-1106',
    # 'gpt-4-0613', 'gpt-4o-2024-11-20', 'o4-mini-2025-04-16', 'o3-2025-04-16', 
    # 'gpt-5-mini-2025-08-07', 'gpt-5-2025-08-07', 'claude-opus-4-1-20250805', 'gemini-2.5-flash'
    # 'mistral-medium-2505', 'Qwen/Qwen3-14B', 'gemini-2.5-flash-lite', 'Llama-3.1-405B-Instruct'
    # 'claude-3-5-haiku-20241022', 'claude-3-haiku-20240307', 'meta-llama/llama-4-scout', 'meta-llama/llama-4-maverick',
    # 'Qwen/Qwen2-7B-Instruct', 'ibm-granite/granite-3.3-8b-base', 'Qwen/Qwen3-32B', 'deepseek-ai/DeepSeek-R1', 
    # 'deepseek-ai/DeepSeek-V3', 'meta-llama/Llama-2-70b-chat-hf', 'meta-llama/Llama-2-7b-chat-hf',  'gemma-3-1b-it',
    # 'ibm-granite/granite-3.3-2b-base', 'claude-3-7-sonnet-20250219', 'claude-sonnet-4-20250514', 'Qwen/Qwen2-72B-Instruct'
    # 'Qwen/Qwen3-235B-A22B-Instruct-2507', 'Qwen/QwQ-32B'
        ]
benchmarks = ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']

print("📊 CHECKING SAVED RESULTS")
print("=" * 60)

total_files = 0
total_completed = 0
total_missing = 0

for model in models:
    model_name = model.replace("/", ".").replace("_", ".")
    
    for bench in benchmarks:
        bench_name = bench.replace("/", ".").replace("_", ".")
        results_dir = Path(f"results/{bench_name}/{model_name}")
        
        if not results_dir.exists():
            print(f"\n⚠️  {bench}: Directory doesn't exist yet")
            continue
        
        # Count files
        json_files = list(results_dir.glob("*.json"))
        completed = 0
        missing_response = 0
        missing_scores = 0
        
        for json_file in json_files:
            total_files += 1
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                    
                if data.get('model_response') == "NA":
                    missing_response += 1
                elif data.get('scores') == "NA":
                    missing_scores += 1
                else:
                    completed += 1
                    total_completed += 1
                    
            except Exception as e:
                print(f"  ⚠️  Error reading {json_file.name}: {e}")
        
        # Summary for this benchmark
        print(f"\n{bench.upper()}:")
        print(f"  Total files: {len(json_files)}")
        print(f"  ✅ Completed: {completed}")
        if missing_response > 0:
            print(f"  ⚠️  Missing responses: {missing_response}")
        if missing_scores > 0:
            print(f"  ⚠️  Missing scores: {missing_scores}")

print("\n" + "=" * 60)
print(f"📊 OVERALL SUMMARY")
print("=" * 60)
print(f"Total result files: {total_files}")
print(f"✅ Fully completed: {total_completed}")
print(f"⚠️  Incomplete: {total_files - total_completed}")

if total_completed == total_files:
    print("\n🎉 ALL RESULTS COMPLETE!")
else:
    print(f"\n⚠️  {total_files - total_completed} results still processing or failed")
    print("   Re-run Cell 9 to retry incomplete items")

### Run this to check if block by safety

In [ ]:
# ============================================================
# VERIFY RESULTS WERE SAVED (WITH BLOCK DETECTION)
# ============================================================

import os
from pathlib import Path
import json

models = ['gemini-2.5-pro']  # Your model
benchmarks = ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']

print("📊 CHECKING SAVED RESULTS")
print("=" * 60)

total_files = 0
total_completed = 0
total_blocked = 0
total_missing = 0

for model in models:
    model_name = model.replace("/", ".").replace("_", ".")
    
    for bench in benchmarks:
        bench_name = bench.replace("/", ".").replace("_", ".")
        results_dir = Path(f"results/{bench_name}/{model_name}")
        
        if not results_dir.exists():
            print(f"\n⚠️  {bench}: Directory doesn't exist yet")
            continue
        
        # Count files
        json_files = list(results_dir.glob("*.json"))
        completed = 0
        blocked = 0
        missing_response = 0
        missing_scores = 0
        
        for json_file in json_files:
            total_files += 1
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                
                # Check for blocked responses
                if data.get('model_response') == "[BLOCKED_BY_SAFETY_FILTER]":
                    blocked += 1
                    total_blocked += 1
                elif data.get('model_response') == "NA":
                    missing_response += 1
                    total_missing += 1
                elif data.get('scores') == "NA":
                    missing_scores += 1
                    total_missing += 1
                else:
                    completed += 1
                    total_completed += 1
                    
            except Exception as e:
                print(f"  ⚠️  Error reading {json_file.name}: {e}")
        
        # Summary for this benchmark
        print(f"\n{bench.upper()}:")
        print(f"  Total files: {len(json_files)}")
        print(f"  ✅ Completed: {completed}")
        if blocked > 0:
            print(f"  🛡️  Blocked by safety: {blocked}")
        if missing_response > 0:
            print(f"  ⚠️  Missing responses: {missing_response}")
        if missing_scores > 0:
            print(f"  ⚠️  Missing scores: {missing_scores}")

print("\n" + "=" * 60)
print(f"📊 OVERALL SUMMARY")
print("=" * 60)
print(f"Total result files: {total_files}")
print(f"✅ Fully completed: {total_completed}")
if total_blocked > 0:
    print(f"🛡️  Blocked by safety filter: {total_blocked}")
if total_missing > 0:
    print(f"⚠️  Incomplete: {total_missing}")

if total_completed == total_files:
    print("\n🎉 ALL RESULTS COMPLETE!")
elif total_blocked > 0:
    print(f"\n⚠️  {total_blocked} responses blocked by Gemini safety filters")
    print("   These are counted as failed responses")
else:
    print(f"\n⚠️  {total_missing} results still processing or failed")
    print("   Re-run processing to retry incomplete items")

# Step 6: Save the results

In [ ]:
# ============================================================
# AGGREGATE RESULTS AND CALCULATE SCORES
# ============================================================

import os
import json
from pathlib import Path
import pandas as pd

models = [
    'gemini-2.5-pro'
    # 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'meta-llama/Llama-3.1-8B-Instruct','meta-llama/Llama-3.1-70B-Instruct',
    # 'gpt-5-nano-2025-08-07', 'gpt-5-2025-08-07-thinking', 'gpt-4o-mini-2024-07-18', 'gpt-3.5-turbo-1106',
    # 'gpt-4-0613', 'gpt-4o-2024-11-20', 'o4-mini-2025-04-16', 'o3-2025-04-16', 
    # 'gpt-5-mini-2025-08-07', 'gpt-5-2025-08-07', 'claude-opus-4-1-20250805', 'gemini-2.5-flash'
    # 'mistral-medium-2505', 'Qwen/Qwen3-14B', 'gemini-2.5-flash-lite', 'Llama-3.1-405B-Instruct'
    # 'claude-3-5-haiku-20241022', 'claude-3-haiku-20240307', 'meta-llama/llama-4-scout', 'meta-llama/llama-4-maverick',
    # 'Qwen/Qwen2-7B-Instruct', 'ibm-granite/granite-3.3-8b-base', 'Qwen/Qwen3-32B', 'deepseek-ai/DeepSeek-R1', 
    # 'deepseek-ai/DeepSeek-V3', 'meta-llama/Llama-2-70b-chat-hf', 'meta-llama/Llama-2-7b-chat-hf',  'gemma-3-1b-it',
    # 'ibm-granite/granite-3.3-2b-base', 'claude-3-7-sonnet-20250219', 'claude-sonnet-4-20250514', 'Qwen/Qwen2-72B-Instruct'
    # 'Qwen/Qwen3-235B-A22B-Instruct-2507', 'Qwen/QwQ-32B'
    ]
benchmarks = ['math', 'musr', 'gpqa', 'mmlu-pro', 'bbh', 'ifeval']

print("📊 CALCULATING BENCHMARK SCORES")
print("=" * 60)

results_summary = []

for model in models:
    model_name = model.replace("/", ".").replace("_", ".")
    
    print(f"\n🤖 Model: {model}")
    print("-" * 60)
    
    for bench in benchmarks:
        bench_name = bench.replace("/", ".").replace("_", ".")
        results_dir = Path(f"results/{bench_name}/{model_name}")
        
        if not results_dir.exists():
            print(f"  {bench:12s}: No results found")
            continue
        
        # Load all result files
        json_files = list(results_dir.glob("*.json"))
        scores_list = []
        
        for json_file in json_files:
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                    
                if data.get('scores') != "NA":
                    scores = data['scores']
                    
                    # Handle different score formats
                    if bench == 'ifeval':
                        # IFEval returns dict with multiple metrics
                        if isinstance(scores, dict):
                            # Use 'prompt_level_strict_acc' as main metric
                            score = scores.get('prompt_level_strict_acc', 0)
                        else:
                            score = 0
                    else:
                        # Other benchmarks return judge scores
                        if isinstance(scores, str):
                            try:
                                scores_dict = json.loads(scores)
                                score = scores_dict.get('correctness_score', 0)
                            except:
                                score = 0
                        elif isinstance(scores, dict):
                            score = scores.get('correctness_score', 0)
                        else:
                            score = 0
                    
                    scores_list.append(score)
                    
            except Exception as e:
                continue
        
        # Calculate average
        if scores_list:
            avg_score = sum(scores_list) / len(scores_list)
            print(f"  {bench:12s}: {avg_score:.3f} ({len(scores_list)} items)")
            
            results_summary.append({
                'model': model,
                'benchmark': bench,
                'score': avg_score,
                'num_items': len(scores_list)
            })
        else:
            print(f"  {bench:12s}: No valid scores yet")

print("\n" + "=" * 60)
print("📊 RESULTS SUMMARY TABLE")
print("=" * 60)

if results_summary:
    df = pd.DataFrame(results_summary)
    
    # Pivot for nice display
    pivot = df.pivot(index='benchmark', columns='model', values='score')
    print("\n")
    print(pivot.to_string())
    
    # Overall average
    print("\n" + "-" * 60)
    print(f"Overall Average: {df['score'].mean():.3f}")
    
    # Save to CSV
    output_file = f"results_summary_{models[0].replace('/', '_')}.csv"
    df.to_csv(output_file, index=False)
    print(f"\n💾 Results saved to: {output_file}")
else:
    print("\n⚠️  No results to display yet")

print("=" * 60)